# EvidenceKD-MedVQA — full pipeline trên Kaggle (giai đoạn 0 → 3 → chấm bảng)

Một notebook chạy lần lượt: **detector** (giai đoạn 0) → vùng ứng viên + giải phẫu +
mẫu âm → **self-test oracle** → **decision branch** giai đoạn 1 → chọn ngưỡng →
**teacher MedGemma**: KD-R + KD-C (nếu attach MedGemma + adapter VinDr) →
giai đoạn 2 (KD-R) → **verifier** + **generator S** (nếu attach flan-T5) →
pipeline suy luận → **Bảng A–C, F** của spec và **Bảng 4–6** của VinDr → zip.

Mỗi giai đoạn tự bỏ qua khi output đã có (resume), nên chạy nhiều phiên:
Save Version → Save & Run All, rồi phiên sau Add Input `evidencekd-full.zip`.

| # | Input | Bắt buộc |
|---|---|---|
| 1 | `evidencekd-kaggle-code.zip` | có |
| 2 | ảnh VinDr: `vpa-repo` hoặc `evidencekd-kaggle-images(-all).zip` | có |
| 3 | `evidencekd-kaggle-weights.zip` (COCO Faster R-CNN) | nên |
| 4 | Kaggle Models → MedGemma 1.5 4B-it (Transformers) | cho teacher / KD |
| 5 | adapter VinDr `medgemma-lora.zip` (output notebook train của VinDR-CXR-VQA) | cho teacher / KD |
| 6 | flan-t5-base (upload thư mục HF làm Dataset) | cho generator S |
| 7 | `evidencekd-full.zip` của phiên trước | để resume |

GPU T4 x2 hoặc P100; Internet có thể tắt. Ước lượng: giai đoạn 0 ≈ 1 phiên;
giai đoạn 1 ≈ 2–3 h; teacher KD ≈ 2–3 phiên (có ngân sách thời gian, tự dừng và resume).

## 0. Setup

In [ ]:
BUNDLE = ""; IMAGES = ""; WEIGHTS = ""
MEDGEMMA = ""        # thư mục MedGemma (config.json + *.safetensors); "" = tự tìm
ADAPTER = ""         # thư mục adapter VinDr (adapter_config.json); "" = tự tìm
GEN_S_MODEL = ""     # thư mục flan-t5-base; "" = tự tìm

SMOKE = None             # ví dụ 64: chạy thử toàn chuỗi trên vài mẫu
DETECTOR_EPOCHS = None   # None = theo config (8)
DECISION_EPOCHS = None   # None = theo config (6)
RETRAIN_DETECTOR = False
RETRAIN_DECISION = False
RETRAIN_GENERATOR = False   # True = train lai verifier + generator S dù outputs/ đã có
RUN_TEACHER = True
KD_SHARDS = 30
KD_R_SUBSET = 0.5        # 0.5 = nửa số ảnh train (design Q3); 1.0 = tất cả
TEACHER_BUDGET_MIN = 420 # phút tối đa dành cho KD-R + KD-C trong phiên này
RUN_BASELINES = True     # chạy MedGemma pretrained + B4 trên CHÍNH split này, để
                         # bảng 4-6 so sánh được cùng chế độ (spec §"Sau giai đoạn 1")
BASELINE_SPLITS = ("test", "val")   # test (300) trước; val (659) để đặt cạnh số paper
BASELINE_BATCH = 4
OFFLINE = None
EPOCHS = None

import os, sys, glob, json, shutil, zipfile, subprocess, time

def find_inputs(marker, root="/kaggle/input", max_depth=6):
    """Every directory under root that contains `marker` (a relative path)."""
    hits = []
    if not os.path.isdir(root):
        return hits
    root_depth = root.rstrip("/").count("/")
    for cur, dirs, files in os.walk(root):
        depth = cur.count("/") - root_depth
        if depth >= max_depth:
            dirs[:] = []
        # never descend into the image folders themselves (thousands of files)
        dirs[:] = [d for d in dirs if d not in ("train", "test")]
        if os.path.exists(os.path.join(cur, marker)):
            hits.append(cur)
    hits.sort(key=lambda p: (p.count("/"), p))
    return hits

def tree(root="/kaggle/input", max_depth=4, limit=60):
    """Print what is attached, shallow, so a wrong attachment is obvious."""
    n = 0
    root_depth = root.rstrip("/").count("/")
    for cur, dirs, files in os.walk(root):
        depth = cur.count("/") - root_depth
        if depth >= max_depth:
            dirs[:] = []
        dirs[:] = [d for d in dirs if d not in ("train", "test")]
        print("  " * depth + os.path.basename(cur) + "/" + ("" if depth else f"   ({cur})"))
        for f in sorted(files)[:5]:
            print("  " * (depth + 1) + f)
        if len(files) > 5:
            print("  " * (depth + 1) + f"... {len(files) - 5} more")
        n += 1
        if n >= limit:
            print("  ..."); break

def unpack(marker, dest="/kaggle/working/unpacked"):
    """Extract any attached .zip whose listing contains `marker`; return the dir."""
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                if not any(n.startswith(marker) or ("/" + marker) in n for n in names):
                    continue
                out = os.path.join(dest, os.path.splitext(os.path.basename(z))[0])
                if not os.path.isdir(out):
                    print(f"unpacking {z} -> {out}")
                    zf.extractall(out)
                return out
        except zipfile.BadZipFile:
            continue
    return None

def locate(marker, what):
    hits = find_inputs(marker)
    if not hits:
        up = unpack(marker)
        if up:
            hits = find_inputs(marker, root=up) or [up]
    if not hits:
        tree()
        raise SystemExit(f"Không tìm thấy {what} (cần thư mục chứa `{marker}`). "
                         f"Hãy Add Input dataset tương ứng rồi chạy lại.")
    return hits[0]

NEEDS = {
    "src/evidencekd/models/detector.py": ["def train", "def predict", "def latest_checkpoint"],
    "src/evidencekd/eval/stage0.py": ["def main", "def _write_vindr_config"],
    "src/evidencekd/eval/vindr_bridge.py": ["def detector_rows", "def oracle_rows", "def grounding_success"],
    "src/evidencekd/eval/detection.py": ["def mean_average_precision"],
    "src/evidencekd/data/detection_dataset.py": ["def build_index", "class VinDrDetectionDataset"],
    "src/evidencekd/data/vindr_processed.py": ["def load_images", "def splits_from_images"],
    "src/evidencekd/preflight.py": ["def run"],
    "src/evidencekd/inference/pipeline.py": ["def run_split", "def resolve_generator_config"],
    "vindr_cxr_vqa/evaluate.py": ["def eval_boxes", "def eval_vqa"],
    "vindr_cxr_vqa/vindr.py": ["def fuse_boxes", "def load_images"],
}

def missing_markers(root):
    out = []
    for rel, snippets in NEEDS.items():
        p = os.path.join(root, rel)
        if not os.path.isfile(p):
            out.append(f"{rel}: file missing"); continue
        src = open(p, encoding="utf-8").read()
        for s in snippets:
            if s not in src:
                out.append(f"{rel}: `{s}` not found")
    return out

NEEDS.update({
    "src/evidencekd/models/decision_branch.py": ["def train", "def predict"],
    "src/evidencekd/data/build_regions.py": ["def main"],
    "src/evidencekd/inference/pipeline.py": ["def run_split", "def tune_thresholds"],
    "src/evidencekd/inference/oracle_decisions.py": ["def write_oracle"],
    "src/evidencekd/eval/tables_spec.py": ["def compute"],
    "src/evidencekd/eval/vindr_tables.py": ["def evaluate"],
    "src/evidencekd/eval/medgemma_baseline.py": ["def baseline_rows", "def adapter_for"],
    "src/evidencekd/teacher/kd_r_targets.py": ["def generate_shard", "def gate_shards"],
    "src/evidencekd/teacher/kd_c_targets.py": ["def generate_shard"],
    "src/evidencekd/models/generator_s.py": ["def train"],
    "src/evidencekd/models/verifier.py": ["def train"],
})

WORK = "/kaggle/working/evidencekd"

CODE_ROOT = BUNDLE or next((h for h in find_inputs("src/evidencekd") if not missing_markers(h)), None)
if CODE_ROOT is None:
    CODE_ROOT = locate("src/evidencekd", "code bundle (evidencekd-kaggle-code.zip)")
bad = missing_markers(CODE_ROOT)
if bad:
    raise SystemExit("Code đính kèm đã CŨ hoặc thiếu — dựng lại bằng "
                     "`python scripts/make_kaggle_bundle.py --part code` rồi upload phiên bản mới:\n  " + "\n  ".join(bad))

PROCESSED_ROOT = locate("data/processed/train.jsonl", "dữ liệu đã dựng (data/processed)")
IMAGES_ROOT = IMAGES or locate("dataset/train_data/train", "ảnh VinDr (dataset/train_data/train)")

WEIGHTS_PATH = WEIGHTS
if not WEIGHTS_PATH:
    cands = glob.glob("/kaggle/input/**/fasterrcnn_resnet50_fpn_coco*.pth", recursive=True)
    if not cands:
        up = unpack("weights/")
        if up:
            cands = glob.glob(os.path.join(up, "**/*.pth"), recursive=True)
    WEIGHTS_PATH = cands[0] if cands else ""

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)
for item in ("src", "configs", "scripts", "vindr_cxr_vqa", "pyproject.toml", "README.md", "docs"):
    s = os.path.join(CODE_ROOT, item)
    if os.path.isdir(s):
        shutil.copytree(s, os.path.join(WORK, item), ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    elif os.path.isfile(s):
        shutil.copy2(s, WORK)
os.chdir(WORK)
for p in (os.path.join(WORK, "src"), WORK):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src") + os.pathsep + WORK + os.pathsep + os.environ.get("PYTHONPATH", "")

n_png = len(glob.glob(os.path.join(IMAGES_ROOT, "dataset/train_data/train/*.png")))
print("CODE_ROOT      ", CODE_ROOT)
print("PROCESSED_ROOT ", PROCESSED_ROOT)
print("IMAGES_ROOT    ", IMAGES_ROOT, f"({n_png} PNG)")
print("WEIGHTS_PATH   ", WEIGHTS_PATH or "(none — backbone from scratch)")
print("WORK           ", WORK)

In [ ]:
import socket

def has_internet(host="pypi.org", port=443, timeout=4):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

OFFLINE = (not has_internet()) if OFFLINE is None else OFFLINE
print("OFFLINE =", OFFLINE)
if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [ ]:
if not OFFLINE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15", "pyyaml"], check=False)
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__,
      "cuda", torch.cuda.is_available(),
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Optional inputs for the later stages. Leave "" to auto-detect; a stage is
# skipped (with a message) when its model is not attached.
def _find_dir(markers, prefer=None):
    for m in markers:
        hits = find_inputs(m)
        if prefer:
            hits = sorted(hits, key=lambda h: (prefer not in h.lower(), h.count("/"), h))
        if hits:
            return hits[0]
    return ""

MEDGEMMA = MEDGEMMA or _find_dir(["config.json"], prefer="medgemma")
if MEDGEMMA and not glob.glob(os.path.join(MEDGEMMA, "*.safetensors")):
    MEDGEMMA = ""
ADAPTER = ADAPTER or next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
                           if "checkpoint-" not in p), "")
if not ADAPTER:
    up = unpack("adapter_config.json")
    if up:
        ADAPTER = next((os.path.dirname(p) for p in glob.glob(os.path.join(up, "**/adapter_config.json"), recursive=True)
                        if "checkpoint-" not in p), "")
GEN_S_MODEL = GEN_S_MODEL or _find_dir(["spiece.model"], prefer="t5")
print("MEDGEMMA   ", MEDGEMMA or "(none -> teacher / KD / generator L skipped)")
print("ADAPTER    ", ADAPTER or "(none -> teacher skipped)")
print("GEN_S_MODEL", GEN_S_MODEL or "(none -> generator S skipped; template generator only)")

## 1. Config cho mọi giai đoạn + khôi phục output phiên trước

In [ ]:
from evidencekd.utils.config import load_config, dump_config
OUT = "/kaggle/working/outputs"; REPORTS = "/kaggle/working/reports"; DERIVED = "/kaggle/working/derived"
CACHE = "/kaggle/working/cache"
for d in (OUT, REPORTS, DERIVED, CACHE):
    os.makedirs(d, exist_ok=True)
# bring back everything a previous session produced (evidencekd-full.zip attached as input)
for sub in ("outputs", "reports", "derived", "cache"):
    for p in glob.glob(f"/kaggle/input/**/evidencekd_full/{sub}", recursive=True):
        shutil.copytree(p, f"/kaggle/working/{sub}", dirs_exist_ok=True); print("restored", p)
for p in glob.glob("/kaggle/input/**/detector/detector.pt", recursive=True):
    os.makedirs(os.path.join(OUT, "detector"), exist_ok=True)
    if not os.path.exists(os.path.join(OUT, "detector", "detector.pt")):
        shutil.copy2(p, os.path.join(OUT, "detector")); print("restored", p)

vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
CFG = {}
common = {"dataset_dir": os.path.join(IMAGES_ROOT, "dataset"), "processed_dir": os.path.join(PROCESSED_ROOT, "data", "processed"),
          "output_dir": OUT, "reports_dir": REPORTS, "derived_dir": DERIVED, "cache_dir": CACHE}
for name in ("detector", "decision", "teacher", "generator"):
    cfg = load_config(os.path.join(WORK, "configs", f"{name}.yaml"))
    for k, v in common.items():
        cfg["paths"][k] = v
    if name == "detector":
        cfg["model"]["coco_weights"] = WEIGHTS_PATH
        cfg["train"]["batch_size"] = 8 if vram >= 14 else 4
        cfg["data"]["image_source"] = "all" if n_png >= 10000 else "vqa_train"
    if name == "decision":
        cfg["paths"]["detector_checkpoint"] = os.path.join(OUT, "detector", "detector.pt")
        cfg["paths"]["kd_r_cache"] = os.path.join(CACHE, "kd_r")
        cfg["train"]["batch_size"] = 16 if vram >= 14 else 8
    if name == "teacher":
        cfg["model"]["name"] = MEDGEMMA or cfg["model"]["name"]
        cfg["model"]["adapter"] = ADAPTER
        cfg["kd_r"]["subset_fraction"] = KD_R_SUBSET
    if name == "generator":
        cfg["generator_s"]["model"] = GEN_S_MODEL or cfg["generator_s"]["model"]
        cfg["generator_l"]["model"] = MEDGEMMA or cfg["generator_l"]["model"]
    CFG[name] = os.path.join(WORK, "configs", f"kaggle_{name}.yaml")
    dump_config(cfg, CFG[name])
print(CFG)
assert os.path.isfile(os.path.join(common["processed_dir"], "train.jsonl"))

def sh(*args, check=True):
    cmd = [sys.executable, "-m", *[str(a) for a in args]]
    print("$", " ".join(cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd)
    print(f"  -> exit {r.returncode} in {(time.time() - t0) / 60:.1f} min", flush=True)
    if check and r.returncode != 0:
        raise SystemExit(f"step failed: {' '.join(cmd)}")
    return r.returncode

## 2. Giai đoạn 0 — detector (train nếu chưa có) + B7 Bảng 4–6

In [ ]:
DET = os.path.join(OUT, "detector", "detector.pt")
if os.path.isfile(DET) and not RETRAIN_DETECTOR:
    print("detector.pt present -> stage 0 training skipped")
else:
    sh("evidencekd.preflight", "--config", CFG["detector"])
    sh("evidencekd.models.detector", "train", "--config", CFG["detector"], "--output-dir", os.path.join(OUT, "detector"),
       "--resume", "auto", *(["--epochs", str(DETECTOR_EPOCHS)] if DETECTOR_EPOCHS else []),
       *(["--limit", str(SMOKE)] if SMOKE else []))
for split in ("val", "test"):
    p = os.path.join(OUT, f"detections_{split}.json")
    if not os.path.isfile(p) or RETRAIN_DETECTOR:
        sh("evidencekd.models.detector", "predict", "--config", CFG["detector"], "--checkpoint", DET, "--split", split, "--out", p)
    sh("evidencekd.eval.stage0", "--config", CFG["detector"], "--detections", p, "--split", split, "--out-dir", REPORTS)

## 3. Vùng ứng viên, giải phẫu, mẫu âm, nhãn — và self-test oracle qua scorer VinDr

In [ ]:
try:
    import torchxrayvision  # noqa: F401
    anat_flag = []
except Exception:
    anat_flag = ["--no-anatomy"]
    print("torchxrayvision not available -> fallback anatomy (no zone/extent facts)")
sh("evidencekd.data.build_regions", "--config", CFG["decision"], *anat_flag)
# self-test of the whole inference chain with the answer key: must be ~perfect
sh("evidencekd.inference.oracle_decisions", "--config", CFG["decision"], "--split", "val")
sh("evidencekd.inference.pipeline", "--config", CFG["decision"], "--split", "val", "--generator", "template",
   "--decision", os.path.join(OUT, "decision_val_oracle.jsonl"), "--regions", os.path.join(DERIVED, "regions_val_annotation.json"), "--tag", "oracle")
sh("evidencekd.eval.vindr_tables", "--config", CFG["decision"], "--pred", os.path.join(REPORTS, "preds_val_oracle.jsonl"), "--split", "val")

## 4. Giai đoạn 1 — decision branch, dự đoán val/test, chọn ngưỡng trên val∖test

In [ ]:
DEC_DIR = os.path.join(OUT, "decision")
S1 = os.path.join(DEC_DIR, "decision_stage1.pt")
if not os.path.isfile(S1) or RETRAIN_DECISION:
    sh("evidencekd.models.decision_branch", "train", "--config", CFG["decision"], "--stage", "1", "--output-dir", DEC_DIR,
       "--resume", "auto", *(["--epochs", str(DECISION_EPOCHS)] if DECISION_EPOCHS else []), *(["--limit", str(SMOKE)] if SMOKE else []))
for split in ("val", "test"):
    sh("evidencekd.models.decision_branch", "predict", "--config", CFG["decision"], "--checkpoint", S1, "--split", split,
       "--out", os.path.join(OUT, f"decision_{split}.jsonl"))
sh("evidencekd.inference.pipeline", "--config", CFG["decision"], "--tune")

## 5. Teacher MedGemma — KD-R (che vùng) và KD-C (rationale), có ngân sách thời gian

In [ ]:
HAVE_TEACHER = bool(MEDGEMMA and ADAPTER) and RUN_TEACHER
if not HAVE_TEACHER:
    print("teacher skipped (needs MedGemma + the VinDr adapter attached, and RUN_TEACHER=True)")
else:
    t_start = time.time()
    def budget_left():
        return (time.time() - t_start) / 60 < TEACHER_BUDGET_MIN

    def run_shards(module, kind):
        """Shards are check=False so one bad sample cannot lose a whole session.
        But when EVERY shard fails the cache stays empty, stage 2 silently
        degrades to stage 1 and the run still exits 0 -- so stop here instead."""
        tried = fails = 0
        for shard in range(KD_SHARDS):
            if not budget_left():
                print(f"{kind}: time budget reached; resume next session"); break
            tried += 1
            rc = sh(module, "--config", CFG["teacher"], "--shard", shard, "--n-shards", KD_SHARDS,
                    *(["--limit", str(SMOKE)] if SMOKE else []), check=False)
            fails += (rc != 0)
        produced = len(glob.glob(os.path.join(CACHE, kind.lower().replace("-", "_"), "*.jsonl")))
        print(f"{kind}: {tried} shard chay, {fails} fail, {produced} file cache")
        if tried and not produced:
            raise SystemExit(
                f"{kind}: TAT CA {tried} shard deu that bai, cache rong. Doc traceback dau tien o tren -- "
                "neu khong dung lai day thi stage 2 se am tham bo qua va ca phien ket thuc exit 0 nhu the da chay.")
        return produced

    run_shards("evidencekd.teacher.kd_r_targets", "KD-R")
    sh("evidencekd.teacher.kd_r_targets", "--config", CFG["teacher"], "--gate", check=False)
    sh("evidencekd.teacher.kd_r_targets", "--config", CFG["teacher"], "--sensitivity", check=False)
    run_shards("evidencekd.teacher.kd_c_targets", "KD-C")

## 6. Giai đoạn 2 — KD-R + faithfulness (khi có cache KD-R)

In [ ]:
FINAL_DEC = S1
if glob.glob(os.path.join(CACHE, "kd_r", "*.jsonl")):
    S2 = os.path.join(DEC_DIR, "decision_stage2.pt")
    if not os.path.isfile(S2) or RETRAIN_DECISION:
        sh("evidencekd.models.decision_branch", "train", "--config", CFG["decision"], "--stage", "2", "--output-dir", DEC_DIR,
           "--init", S1, "--resume", "auto", *(["--limit", str(SMOKE)] if SMOKE else []))
    for split in ("val", "test"):
        sh("evidencekd.models.decision_branch", "predict", "--config", CFG["decision"], "--checkpoint", S2, "--split", split,
           "--out", os.path.join(OUT, f"decision_{split}.jsonl"))
    sh("evidencekd.inference.pipeline", "--config", CFG["decision"], "--tune")
    FINAL_DEC = S2
else:
    print("no KD-R cache -> stage 2 skipped; the stage-1 decision branch is evaluated")
print("decision checkpoint for evaluation:", FINAL_DEC)

## 7. Giai đoạn 3 — verifier học (khi có KD-C) và generator S (khi có flan-T5)

In [ ]:
VERIFIER = ""
GEN_S = ""
V_PT = os.path.join(OUT, "verifier", "verifier.pt")
GS_FINAL = os.path.join(OUT, "generator_s", "final")
if os.path.isfile(V_PT) and not RETRAIN_GENERATOR:
    VERIFIER = V_PT; print("verifier.pt da co -> bo qua train")
elif glob.glob(os.path.join(CACHE, "kd_c", "*.jsonl")):
    sh("evidencekd.models.verifier", "train", "--config", CFG["generator"], "--output-dir", os.path.join(OUT, "verifier"), check=False)
    if os.path.isfile(V_PT):
        VERIFIER = V_PT
if os.path.isdir(GS_FINAL) and not RETRAIN_GENERATOR:
    GEN_S = GS_FINAL; print("generator_s/final da co -> bo qua train")
elif GEN_S_MODEL:
    rc = sh("evidencekd.models.generator_s", "train", "--config", CFG["generator"], "--output-dir", os.path.join(OUT, "generator_s"),
            *(["--limit", str(SMOKE)] if SMOKE else []), check=False)
    if rc == 0 and os.path.isdir(GS_FINAL):
        GEN_S = GS_FINAL
print("verifier:", VERIFIER or "rules only", "| generator S:", GEN_S or "not trained (template generator)")

## 8. Baseline MedGemma trên chính split này — pretrained và B4 (fine-tuned)

In [ ]:
# B4 (MedGemma + adapter VinDr) va pretrained (cung base, khong adapter) CHAY TREN
# CHINH SPLIT nay -- de "co cai thien khong" tra loi duoc, thay vi dat canh hang so
# val-659 cua paper. Ton kem: ~959 sinh/tag cho ca hai split, nen co resume.
BASELINES = []
if not RUN_BASELINES:
    print("baselines bi tat (RUN_BASELINES=False)")
elif not MEDGEMMA:
    print("khong co MedGemma -> khong co B4/pretrained; bang so sanh chi con hang so val-659 cua paper")
else:
    BASELINES = ["pretrained"] + (["b4"] if ADAPTER else [])
    if not ADAPTER:
        print("co MedGemma nhung thieu adapter -> chi chay pretrained, KHONG co B4")
    for split in BASELINE_SPLITS:
        for tag in BASELINES:
            sh("evidencekd.eval.medgemma_baseline", "--config", CFG["teacher"], "--split", split, "--tag", tag,
               "--batch-size", BASELINE_BATCH, *(["--limit", str(SMOKE)] if SMOKE else []), check=False)
print("baselines:", BASELINES or "(none)")

## 9. Suy luận Proposed-S và chấm: Bảng A–C, F (spec) + Bảng 4–6 (VinDr) + faithfulness

In [ ]:
preds = {}
for split in ("val", "test"):
    gens = [("template", [])] + ([("s", ["--generator-dir", GEN_S, "--generator-config", CFG["generator"]])] if GEN_S else [])
    for gen, extra in gens:
        sh("evidencekd.inference.pipeline", "--config", CFG["decision"], "--split", split, "--generator", gen, *extra,
           *(["--verifier", VERIFIER] if VERIFIER else []))
        sh("evidencekd.eval.tables_spec", "--config", CFG["decision"], "--pipeline", os.path.join(OUT, f"pipeline_{split}_{gen}.jsonl"), "--split", split)
        preds.setdefault(split, []).append(os.path.join(REPORTS, f"preds_{split}_{gen}.jsonl"))
    # B7 (ablation noi bo) va hai baseline MedGemma, deu tren split nay
    for tag in ("b7", "pretrained", "b4"):
        p = os.path.join(REPORTS, f"preds_{split}_{tag}.jsonl")
        if os.path.isfile(p):
            preds[split].append(p)
    sh("evidencekd.eval.vindr_tables", "--config", CFG["decision"], "--pred", *preds[split], "--split", split, check=False)
sh("evidencekd.eval.faithfulness", "--config", CFG["decision"], "--checkpoint", FINAL_DEC,
   "--pipeline", os.path.join(OUT, "pipeline_test_template.jsonl"), "--split", "test", "--limit", "300", check=False)

## 10. Lưu output để resume / tải về

In [ ]:
stage = "/kaggle/working/evidencekd_full"
shutil.rmtree(stage, ignore_errors=True)
for sub in ("reports", "derived", "cache"):
    if os.path.isdir(f"/kaggle/working/{sub}"):
        shutil.copytree(f"/kaggle/working/{sub}", os.path.join(stage, sub))
os.makedirs(os.path.join(stage, "outputs"), exist_ok=True)
keep = ["detector/detector.pt", "decision/decision_stage1.pt", "decision/decision_stage2.pt", "decision/vocab.json",
        "verifier/verifier.pt", "generator_s/final", "detections_val.json", "detections_test.json",
        "decision_val.jsonl", "decision_test.jsonl"] + [os.path.basename(p) for p in glob.glob(os.path.join(OUT, "pipeline_*.jsonl"))]
for k in keep:
    p = os.path.join(OUT, k)
    if os.path.isdir(p):
        shutil.copytree(p, os.path.join(stage, "outputs", k), dirs_exist_ok=True)
    elif os.path.isfile(p):
        os.makedirs(os.path.dirname(os.path.join(stage, "outputs", k)), exist_ok=True); shutil.copy2(p, os.path.join(stage, "outputs", k))
zip_path = shutil.make_archive("/kaggle/working/evidencekd-full", "zip", "/kaggle/working", "evidencekd_full")
print(f"{os.path.getsize(zip_path) / 2**20:.0f} MB ->", zip_path)
print("Phiên sau: Add Input file này để resume mọi giai đoạn (checkpoint, cache KD, report).")
for p in sorted(glob.glob(os.path.join(REPORTS, "*.json"))):
    print(" ", os.path.basename(p))